# Pratyaksha — Python data structures, rendered in the notebook

Build a data structure in Python and see it drawn in the output cell. State is
driven from your code: mutate the object, display it again, and the drawing
follows.

**Run the cells in order.** Cell 2 matters on Colab and is easy to skip.

### What is real today

`Stack` and `Queue` have renderers built for the notebook. Twelve other
structures are registered and sync their state, but they render by mounting the
full web-app component, which was built for a page rather than a cell — they
work, they are just not tailored yet.

There are **no animations** here. The widget applies state directly; the
animated stepping lives in the web app at stack-n-flow.

## 1. Install

In [ ]:
# Colab ships ipywidgets 7.7.1. anywidget's front end needs the ipywidgets 8
# protocol, so on 7.x the widget installs, emits a correct mimebundle, and then
# renders nothing at all with no error. --upgrade is what forces it up.
!pip install --quiet --upgrade "ipywidgets>=8" "git+https://github.com/ShivamMalge/Stack-n-Flow.git"

## 2. Restart the runtime

**This step is required on Colab and cannot be skipped.**

The front end already loaded the old widget manager when the notebook opened.
Upgrading `ipywidgets` on disk does not change what is already running, so the
page has to be reloaded before anything can render.

**Runtime -> Restart session**, then carry on from the next cell. Do not re-run
the install cell.

On Jupyter or JupyterLab there is nothing to restart; skip straight on.

## 3. Check the front end can draw

In [ ]:
# Two things have to be true before any widget can draw. This cell checks both
# and says which one is wrong, rather than leaving a blank output to interpret.
import ipywidgets

major = int(ipywidgets.__version__.split('.')[0])
if major < 8:
    raise SystemExit(
        f'ipywidgets is {ipywidgets.__version__}, and anywidget needs 8 or newer. '
        'Restart the runtime (Runtime -> Restart session) and run this cell again. '
        'If it still reports 7.x after a restart, re-run the install cell, then restart.'
    )

print(f'ipywidgets {ipywidgets.__version__} - ok')

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print('Colab: custom widget manager enabled')
except ImportError:
    print('Not running on Colab - nothing to enable')

## 4. Stack

Push three values, then display. The last one pushed sits on top.

In [ ]:
from pratyaksha import Stack

s = Stack()
for value in (10, 20, 30):
    s.push(value)

s

Now pop one and display again — the drawing follows the Python object.

In [ ]:
s.pop()
s

## 5. Queue

First in, first out: `A` is at the front and leaves first.

In [ ]:
from pratyaksha import Queue

q = Queue()
for label in ("A", "B", "C"):
    q.enqueue(label)

q

In [ ]:
q.dequeue()
q

## 6. One of the twelve that reuse the web-app component

An AVL tree. It syncs correctly, but the component it mounts was laid out for a
full page, so expect it to look cramped in a notebook cell. Extracting proper
notebook renderers for these is the next phase of work.

In [ ]:
from pratyaksha import AVLTree

tree = AVLTree()
tree.set_root(50)
tree

## 7. If nothing rendered

Run this and send the output — it distinguishes a packaging problem from a
widget-manager one, which look identical from the outside.

In [ ]:
import pathlib, sys
import pratyaksha

print("python       ", sys.version.split()[0])
print("pratyaksha   ", pratyaksha.__version__)

static = pathlib.Path(pratyaksha.__file__).parent / "static"
for name in ("pratyaksha-bridge.mjs", "pratyaksha.css"):
    asset = static / name
    size = asset.stat().st_size if asset.exists() else 0
    print(f"  {name:24} {'present' if asset.exists() else 'MISSING'}  {size:,} bytes")

try:
    import anywidget, ipywidgets
    major = int(ipywidgets.__version__.split(".")[0])
    print("anywidget    ", anywidget.__version__)
    print("ipywidgets   ", ipywidgets.__version__,
          "- ok" if major >= 8 else "- TOO OLD, anywidget needs 8+. Restart the runtime.")
except ImportError as exc:
    print("MISSING dependency:", exc)

from pratyaksha import Stack
probe = Stack()
probe.push(1)
bundle = probe._repr_mimebundle_()
data = bundle[0] if isinstance(bundle, tuple) else bundle
print("mime types   ", sorted(data))
print("nodes synced ", len(probe.widget.nodes))
print()
print("If the mime type and node count are right but nothing drew, the Python")
print("side is fine and the problem is the front end - almost always ipywidgets 7")
print("still being live because the runtime was not restarted.")